# SahajMobile · Cohort-Based MOIC Curve Analysis

Build a cohort-based MOIC (multiple on invested capital) curve from raw installment payment data.

**Formula reference**

| Metric | Formula |
|---|---|
| Payment MOIC | Period collections ÷ Cohort total advance |
| Cumulative MOIC | Running collections ÷ Cohort total advance |

**Workflow** — the blocks below clean the data, calculate the cohort summary tables, and write the deliverables to disk.

## Setup — imports & configuration

Same code as the top of the `.py` file. In a notebook, `__file__` does not exist, so paths resolve from the current working directory (run this notebook from the folder that contains the CSV).

In [1]:
%matplotlib inline
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

# ─── Configuration ─────────────────────────────────────────────────────────────
# Resolution order for input/output paths: env vars → defaults next to this file.
# Run this notebook from the project folder that contains the CSV.
BASE_DIR   = Path(".").resolve()
INPUT_PATH = os.environ.get(
    "SAHAJMOBILE_INPUT", str(BASE_DIR / "Installment_shorter_sampled.csv")
)
OUTPUT_DIR = os.environ.get("SAHAJMOBILE_OUTPUT", str(BASE_DIR / "outputs"))

# Payments > MAX_MOB months after origination are flagged as anomalous.
# Chosen as 24 months (generous upper bound for a phone EMI loan).
MAX_MOB = 24


def resolve_paths() -> tuple[str, str]:
    """Resolve (input_csv, output_dir): env vars → defaults next to this file."""
    in_path  = INPUT_PATH
    out_path = OUTPUT_DIR

    if not os.path.isfile(in_path):
        print(f"[Error] Input file not found: {in_path}")
        print(f"  Set SAHAJMOBILE_INPUT to point at the CSV, e.g.:")
        print(f"  %env SAHAJMOBILE_INPUT=/full/path/to/Installment_shorter_sampled.csv")
        sys.exit(1)

    os.makedirs(out_path, exist_ok=True)
    return in_path, out_path


# Leading characters that spreadsheet applications interpret as a formula.
_FORMULA_PREFIXES = ("=", "+", "-", "@", "\t", "\r")


def _neutralize_formula(value):
    """Prefix spreadsheet-formula triggers with a quote so Excel treats them as text."""
    if isinstance(value, str) and value.startswith(_FORMULA_PREFIXES):
        return "'" + value
    return value


def write_csv(df: pd.DataFrame, path: str) -> None:
    """Write a DataFrame to CSV with untrusted text neutralized against CSV injection."""
    safe = df.copy()
    for column in safe.columns:
        if safe[column].dtype == object:
            safe[column] = safe[column].map(_neutralize_formula)
    safe.to_csv(path, index=False)


in_path, out_path = resolve_paths()
print(f"  Input CSV : {in_path}")
print(f"  Output dir: {out_path}")

  Input CSV : /Users/dipro/Task Sahaj Mobile/Installment_shorter_sampled.csv
  Output dir: /Users/dipro/Task Sahaj Mobile/outputs


## Section 1 · Load & Clean

Standardise column names and types, remove exact duplicates, and separate anomalous rows.

In [2]:
def _parse_money(series: pd.Series) -> pd.Series:
    """
    Convert messy currency strings to float.

    Examples
    --------
    ' 17,160 ' → 17160.0
    ' - '      →     0.0   (dash = scheduled but unpaid)
    ' 9,000 '  →  9000.0
    NaN        →     NaN
    """
    def _conv(v):
        if pd.isna(v):
            return np.nan
        s = str(v).strip().replace(",", "").replace(" ", "")
        # Strip all dash variants (unicode en-dash, em-dash, minus, hyphen)
        for ch in ("\u2014", "\u2013", "\u2212", "-"):
            s = s.replace(ch, "")
        if s in ("", "."):
            return 0.0
        try:
            return float(s)
        except ValueError:
            return np.nan

    return series.map(_conv)


def load_and_clean(filepath: str) -> tuple:
    """
    Load the raw CSV, standardise column names and data types, remove exact
    duplicates, and separate anomalous rows.

    Returns
    -------
    cleaned  : pd.DataFrame – analysis-ready rows with added derived columns
    excluded : pd.DataFrame – removed rows with Exclusion_Reason annotation
    """
    # ── Load ──────────────────────────────────────────────────────────────────
    raw = pd.read_csv(filepath)
    raw.columns = raw.columns.str.strip()
    raw = raw.rename(
        columns={
            "Asset ID":         "Asset_ID",
            "Origination Date": "Origination_Date",
            "Total Advance":    "Total_Advance",
            "Total EMI":        "Total_EMI",
            "Payment Date":     "Payment_Date",
            "Payment Amount":   "Payment_Amount",
        }
    )

    # ── Parse numeric columns ─────────────────────────────────────────────────
    raw["Total_Advance"]  = _parse_money(raw["Total_Advance"])
    raw["Total_EMI"]      = _parse_money(raw["Total_EMI"])
    raw["Payment_Amount"] = _parse_money(raw["Payment_Amount"])

    # ── Parse date columns ────────────────────────────────────────────────────
    raw["Origination_Date"] = pd.to_datetime(
        raw["Origination_Date"], errors="coerce", dayfirst=False
    )
    raw["Payment_Date"] = pd.to_datetime(
        raw["Payment_Date"], errors="coerce", dayfirst=False
    )

    # ── Step 1: Remove exact duplicate rows ───────────────────────────────────
    n_raw   = len(raw)
    raw     = raw.drop_duplicates()
    n_dupes = n_raw - len(raw)

    # ── Step 2: Build exclusion masks (first match wins per row) ──────────────
    EPOCH    = pd.Timestamp("1970-01-01")
    both     = raw["Payment_Date"].notna() & raw["Origination_Date"].notna()

    def _mob_series(df):
        """Calendar-month distance: Payment_Date - Origination_Date (in months)."""
        return (
            (df["Payment_Date"].dt.year  - df["Origination_Date"].dt.year)  * 12
            + (df["Payment_Date"].dt.month - df["Origination_Date"].dt.month)
        )

    masks = {
        "Unparseable Origination_Date":
            raw["Origination_Date"].isna(),

        "Unparseable Payment_Date":
            raw["Payment_Date"].isna(),

        "Sentinel / Epoch Payment_Date (1970-01-01)":
            raw["Payment_Date"] == EPOCH,

        "Payment_Date before Origination_Date":
            both & (raw["Payment_Date"] < raw["Origination_Date"]),

        f"Months on Book > {MAX_MOB} — likely data entry error":
            both & (_mob_series(raw) > MAX_MOB),

        "Missing or non-positive Total_Advance":
            raw["Total_Advance"].isna() | (raw["Total_Advance"] <= 0),
    }

    # First-match exclusion: each row gets at most one reason label
    excl_idx  = []
    excl_why  = []
    seen      = set()
    for reason, mask in masks.items():
        new_idx = [i for i in raw.index[mask] if i not in seen]
        excl_idx.extend(new_idx)
        excl_why.extend([reason] * len(new_idx))
        seen.update(new_idx)

    excluded = raw.loc[excl_idx].copy()
    excluded["Exclusion_Reason"] = excl_why

    # ── Step 3: Build cleaned dataset ─────────────────────────────────────────
    cleaned = raw.drop(index=list(seen)).copy()

    # Treat remaining NaN payment amounts as 0 (scheduled but unpaid)
    cleaned["Payment_Amount"] = cleaned["Payment_Amount"].fillna(0.0)

    # Derived columns
    cleaned["Months_on_Book"] = (
        _mob_series(cleaned).clip(lower=0).astype(int)
    )
    cleaned["Cohort"] = (
        cleaned["Origination_Date"].dt.to_period("M").astype(str)
    )

    # ── Console diagnostics ───────────────────────────────────────────────────
    print(f"  Raw rows            : {n_raw:>7,}")
    print(f"  Exact duplicates    : {n_dupes:>7,}  (removed)")
    print(f"  Anomalous/excluded  : {len(excluded):>7,}  rows")
    print(f"  Cleaned rows        : {len(cleaned):>7,}")
    print(f"  Unique cohorts      : {cleaned['Cohort'].nunique():>7}")
    print(f"  Unique assets       : {cleaned['Asset_ID'].nunique():>7}")

    if len(excluded):
        print("\n  Exclusion breakdown:")
        for reason, cnt in excluded["Exclusion_Reason"].value_counts().items():
            print(f"    {cnt:>4}  {reason}")

    return cleaned, excluded


cleaned, excluded = load_and_clean(in_path)

print("\n── Cleaned dataset (first 10 rows) ──────────────────────────")
display(cleaned.head(10))

if len(excluded):
    print("\n── Excluded rows (with reason) ────────────────────────────")
    display(excluded[["Asset_ID", "Origination_Date", "Payment_Date", "Payment_Amount", "Exclusion_Reason"]])

  Raw rows            :  11,894
  Exact duplicates    :     638  (removed)
  Anomalous/excluded  :       4  rows
  Cleaned rows        :  11,252
  Unique cohorts      :      15
  Unique assets       :     351

  Exclusion breakdown:
       2  Payment_Date before Origination_Date
       1  Sentinel / Epoch Payment_Date (1970-01-01)
       1  Months on Book > 24 — likely data entry error

── Cleaned dataset (first 10 rows) ──────────────────────────


,Asset_ID,Origination_Date,Total_Advance,Total_EMI,Payment_Date,Payment_Amount,Months_on_Book,Cohort
0,264,2024-01-31,17160.0,20540.0,2024-02-07,860.0,1,2024-01
1,264,2024-01-31,17160.0,20540.0,2024-02-14,860.0,1,2024-01
2,264,2024-01-31,17160.0,20540.0,2024-02-21,860.0,1,2024-01
3,264,2024-01-31,17160.0,20540.0,2024-02-28,860.0,1,2024-01
4,264,2024-01-31,17160.0,20540.0,2024-03-06,860.0,2,2024-01
5,264,2024-01-31,17160.0,20540.0,2024-03-13,860.0,2,2024-01
6,264,2024-01-31,17160.0,20540.0,2024-03-20,860.0,2,2024-01
7,264,2024-01-31,17160.0,20540.0,2024-03-27,860.0,2,2024-01
8,264,2024-01-31,17160.0,20540.0,2024-04-03,860.0,3,2024-01
9,264,2024-01-31,17160.0,20540.0,2024-04-10,9000.0,3,2024-01



── Excluded rows (with reason) ────────────────────────────


,Asset_ID,Origination_Date,Payment_Date,Payment_Amount,Exclusion_Reason
11892,24105,2026-01-06,1970-01-01,2500.0,Sentinel / Epoch Payment_Date (1970-01-01)
10085,1454,2024-11-18,1970-01-03,1.0,Payment_Date before Origination_Date
10207,1465,2024-11-20,2024-07-02,0.0,Payment_Date before Origination_Date
11888,13747,2025-10-24,2027-11-27,2000.0,Months on Book > 24 — likely data entry error


## Section 2 · MOIC Computation

Aggregate collections by cohort × months-on-book; compute Payment MOIC and Cumulative MOIC.

In [3]:
def compute_moic(cleaned: pd.DataFrame) -> tuple:
    """
    Aggregate collections by cohort × months-on-book and compute:

      Payment_MOIC    = Period_Collections   / Cohort_Total_Advance
      Cumulative_MOIC = Running_Collections  / Cohort_Total_Advance

    Design decisions
    ----------------
    1. Only rows with Payment_Amount > 0 feed into MOIC aggregation.
       Zero-amount rows (scheduled-but-unpaid instalment slots) are excluded
       so future scheduled slots don't extend the curve with spurious zeros.
    2. After aggregation, missing MOB months WITHIN each cohort's observed range
       are forward-filled with Period_Collections = 0 so the cumulative curve
       shows flat segments (no collection) instead of omitting those months.

    Returns
    -------
    moic_df     : pd.DataFrame – long form MOIC table (cohort × MOB)
    cohort_meta : pd.DataFrame – cohort-level advance & asset count
    """
    # ── Per-asset advance (constant per asset; take first occurrence) ─────────
    asset_meta = (
        cleaned.groupby("Asset_ID")
               .agg(Cohort        = ("Cohort",         "first"),
                    Total_Advance = ("Total_Advance",  "first"))
               .reset_index()
    )

    # ── Cohort-level metadata ─────────────────────────────────────────────────
    cohort_meta = (
        asset_meta.groupby("Cohort")
                  .agg(Asset_Count          = ("Asset_ID",      "count"),
                       Cohort_Total_Advance = ("Total_Advance", "sum"))
                  .reset_index()
    )

    # ── Aggregate actual (positive) payments only ─────────────────────────────
    paid = cleaned[cleaned["Payment_Amount"] > 0].copy()

    agg = (
        paid.groupby(["Cohort", "Months_on_Book"])["Payment_Amount"]
            .sum()
            .reset_index()
            .rename(columns={"Payment_Amount": "Period_Collections"})
    )
    agg = agg.sort_values(["Cohort", "Months_on_Book"]).reset_index(drop=True)

    # ── Fill missing MOB months within each cohort's observed range ───────────
    ranges = agg.groupby("Cohort")["Months_on_Book"].agg(["min", "max"]).reset_index()
    full_grid = pd.concat(
        [
            pd.DataFrame({
                "Cohort":         row.Cohort,
                "Months_on_Book": range(int(row["min"]), int(row["max"]) + 1),
            })
            for _, row in ranges.iterrows()
        ],
        ignore_index=True,
    )
    agg = full_grid.merge(agg, on=["Cohort", "Months_on_Book"], how="left")
    agg["Period_Collections"] = agg["Period_Collections"].fillna(0.0)

    # ── Merge cohort advance ──────────────────────────────────────────────────
    agg = agg.merge(
        cohort_meta[["Cohort", "Cohort_Total_Advance"]], on="Cohort", how="left"
    )
    agg = agg.sort_values(["Cohort", "Months_on_Book"]).reset_index(drop=True)

    # ── MOIC metrics ──────────────────────────────────────────────────────────
    agg["Payment_MOIC"] = (
        agg["Period_Collections"] / agg["Cohort_Total_Advance"]
    )
    agg["Cumulative_Collections"] = (
        agg.groupby("Cohort")["Period_Collections"].cumsum()
    )
    agg["Cumulative_MOIC"] = (
        agg["Cumulative_Collections"] / agg["Cohort_Total_Advance"]
    )
    agg["Net_MOIC"] = (
        agg["Cumulative_MOIC"] - 1.0
    )

    print(f"  Cohorts in MOIC table : {agg['Cohort'].nunique()}")
    print(f"  Max Months on Book    : {agg['Months_on_Book'].max()}")
    print(
        f"  Max Cumulative MOIC   : "
        f"{agg['Cumulative_MOIC'].max():.4f}×"
        f"  (cohort {agg.loc[agg['Cumulative_MOIC'].idxmax(), 'Cohort']})"
    )
    print(
        f"  Max Net MOIC          : "
        f"{agg['Net_MOIC'].max():+.4f}×"
        f"  (cohort {agg.loc[agg['Net_MOIC'].idxmax(), 'Cohort']})"
    )

    return agg, cohort_meta


moic_df, cohort_meta = compute_moic(cleaned)

print("\n── Cohort metadata ──────────────────────────────────────────")
display(cohort_meta)

print("\n── MOIC long-form table (first 15 rows) ────────────────────")
display(moic_df.head(15))

  Cohorts in MOIC table : 15
  Max Months on Book    : 21
  Max Cumulative MOIC   : 1.3903×  (cohort 2024-09)
  Max Net MOIC          : +0.3903×  (cohort 2024-09)

── Cohort metadata ──────────────────────────────────────────


,Cohort,Asset_Count,Cohort_Total_Advance
0,2024-01,6,102945.0
1,2024-02,11,136095.0
2,2024-03,64,795665.0
3,2024-04,32,368935.0
4,2024-05,16,224805.0
5,2024-06,32,447855.0
6,2024-07,20,286900.0
7,2024-08,28,325990.0
8,2024-09,37,408306.0
9,2024-10,37,461204.0



── MOIC long-form table (first 15 rows) ────────────────────


,Cohort,Months_on_Book,Period_Collections,Cohort_Total_Advance,Payment_MOIC,Cumulative_Collections,Cumulative_MOIC,Net_MOIC
0,2024-01,1,23498.0,102945.0,0.228258,23498.0,0.228258,-0.771742
1,2024-01,2,24097.0,102945.0,0.234076,47595.0,0.462334,-0.537666
2,2024-01,3,39580.0,102945.0,0.384477,87175.0,0.846811,-0.153189
3,2024-01,4,12689.0,102945.0,0.123260,99864.0,0.970071,-0.029929
4,2024-01,5,11843.0,102945.0,0.115042,111707.0,1.085113,0.085113
5,2024-01,6,4950.0,102945.0,0.048084,116657.0,1.133197,0.133197
6,2024-01,7,0.0,102945.0,0.000000,116657.0,1.133197,0.133197
7,2024-01,8,0.0,102945.0,0.000000,116657.0,1.133197,0.133197
8,2024-01,9,2493.0,102945.0,0.024217,119150.0,1.157414,0.157414
9,2024-02,0,7520.0,136095.0,0.055256,7520.0,0.055256,-0.944744


## Section 3 · Summary Tables

Cohort KPI summary plus wide pivots of Cumulative and Payment MOIC.

In [4]:
def build_tables(moic_df: pd.DataFrame, cohort_meta: pd.DataFrame) -> tuple:
    """
    Construct three deliverable tables.

    Returns
    -------
    summary     : pd.DataFrame – one-row-per-cohort KPI summary
    cum_matrix  : pd.DataFrame – wide pivot  cohort × MOB → Cumulative MOIC
    pay_matrix  : pd.DataFrame – wide pivot  cohort × MOB → Payment MOIC
    """
    # ── Cohort KPI summary ────────────────────────────────────────────────────
    kpis = (
        moic_df.groupby("Cohort")
               .agg(
                   Total_Collected     = ("Period_Collections", "sum"),
                   Max_Months_on_Book  = ("Months_on_Book",     "max"),
                   Max_Cumulative_MOIC = ("Cumulative_MOIC",    "max"),
                   Max_Net_MOIC        = ("Net_MOIC",           "max"),
                   Avg_Payment_MOIC    = ("Payment_MOIC",       "mean"),
               )
               .reset_index()
    )
    summary = cohort_meta.merge(kpis, on="Cohort", how="left").sort_values("Cohort")

    # Reorder for readability
    summary = summary[[
        "Cohort", "Asset_Count", "Cohort_Total_Advance",
        "Total_Collected", "Max_Months_on_Book",
        "Max_Cumulative_MOIC", "Max_Net_MOIC", "Avg_Payment_MOIC",
    ]]

    # ── Pivot: Cumulative MOIC ────────────────────────────────────────────────
    cum_matrix = (
        moic_df.pivot_table(
            index="Cohort", columns="Months_on_Book",
            values="Cumulative_MOIC", aggfunc="first",
        )
        .reset_index()
    )
    cum_matrix.columns.name = None
    cum_matrix.columns = (
        ["Cohort"] + [f"MOB_{c}" for c in cum_matrix.columns[1:]]
    )

    # ── Pivot: Payment MOIC ───────────────────────────────────────────────────
    pay_matrix = (
        moic_df.pivot_table(
            index="Cohort", columns="Months_on_Book",
            values="Payment_MOIC", aggfunc="first",
        )
        .reset_index()
    )
    pay_matrix.columns.name = None
    pay_matrix.columns = (
        ["Cohort"] + [f"MOB_{c}" for c in pay_matrix.columns[1:]]
    )

    return summary, cum_matrix, pay_matrix


def build_executive_insights(moic_df: pd.DataFrame, cohort_summary: pd.DataFrame) -> tuple:
    """
    Build a CTO-facing cohort analysis table with breakeven timing and recovery flags.
    """
    latest_rows = (
        moic_df.sort_values(["Cohort", "Months_on_Book"])
              .groupby("Cohort", as_index=False)
              .tail(1)
              [["Cohort", "Months_on_Book", "Cumulative_MOIC", "Payment_MOIC"]]
              .rename(columns={
                  "Months_on_Book": "Latest_MOB",
                  "Cumulative_MOIC": "Latest_Cumulative_MOIC",
                  "Payment_MOIC": "Latest_Payment_MOIC",
              })
    )

    def _first_crossing(group: pd.DataFrame, threshold: float):
        crossed = group.loc[group["Cumulative_MOIC"] >= threshold, "Months_on_Book"]
        return int(crossed.iloc[0]) if not crossed.empty else np.nan

    threshold_map = (
        moic_df.sort_values(["Cohort", "Months_on_Book"])
              .groupby("Cohort")
              .apply(lambda group: pd.Series({
                  "Months_to_Breakeven": _first_crossing(group, 1.0),
                  "Months_to_Target_1_3x": _first_crossing(group, 1.3),
              }))
              .reset_index()
    )

    insights = cohort_summary.merge(latest_rows, on="Cohort", how="left")
    insights = insights.merge(threshold_map, on="Cohort", how="left")
    insights["Collection_Share_Pct"] = (
        insights["Total_Collected"] / insights["Total_Collected"].sum() * 100
    )
    insights["Advance_Share_Pct"] = (
        insights["Cohort_Total_Advance"] / insights["Cohort_Total_Advance"].sum() * 100
    )
    insights["Seasoned_12M"] = insights["Max_Months_on_Book"] >= 12
    insights["Recovery_Status"] = np.select(
        [insights["Latest_Cumulative_MOIC"] >= 1.3, insights["Latest_Cumulative_MOIC"] >= 1.0],
        ["Beyond target", "Recovered"],
        default="Below breakeven",
    )
    insights["Months_to_Breakeven"] = insights["Months_to_Breakeven"].astype("Int64")
    insights["Months_to_Target_1_3x"] = insights["Months_to_Target_1_3x"].astype("Int64")

    insights = insights[[
        "Cohort",
        "Recovery_Status",
        "Seasoned_12M",
        "Latest_MOB",
        "Latest_Cumulative_MOIC",
        "Months_to_Breakeven",
        "Months_to_Target_1_3x",
        "Collection_Share_Pct",
        "Advance_Share_Pct",
        "Max_Cumulative_MOIC",
    ]].sort_values(["Latest_Cumulative_MOIC", "Collection_Share_Pct"], ascending=[False, False])

    portfolio_rate = cohort_summary["Total_Collected"].sum() / cohort_summary["Cohort_Total_Advance"].sum()
    top3_share = insights.head(3)["Collection_Share_Pct"].sum()
    recovered_share = insights.loc[insights["Latest_Cumulative_MOIC"] >= 1.0, "Collection_Share_Pct"].sum()
    portfolio_summary = pd.DataFrame([
        {"Metric": "Portfolio Recovery Rate", "Value": round(portfolio_rate, 4), "Unit": "x"},
        {"Metric": "Recovered Cohort Share", "Value": round(recovered_share, 2), "Unit": "% of collections"},
        {"Metric": "Top-3 Cohort Share", "Value": round(top3_share, 2), "Unit": "% of collections"},
        {"Metric": "Seasoned Cohorts (12M+)", "Value": int(insights["Seasoned_12M"].sum()), "Unit": "cohorts"},
        {"Metric": "Cohorts Above 1.0x", "Value": int((insights["Latest_Cumulative_MOIC"] >= 1.0).sum()), "Unit": "cohorts"},
        {"Metric": "Cohorts Above 1.3x", "Value": int((insights["Latest_Cumulative_MOIC"] >= 1.3).sum()), "Unit": "cohorts"},
    ])

    return insights, portfolio_summary


summary, cum_matrix, pay_matrix = build_tables(moic_df, cohort_meta)
print("── Cohort summary table ────────────────────────────────────")
display(summary)

print("\n── Cumulative MOIC matrix (cohort × MOB) ───────────────────")
display(cum_matrix)

print("\n── Payment MOIC matrix (cohort × MOB) ─────────────────────")
display(pay_matrix)

print("\n[3b] Building Executive Insights ...")
executive_insights, portfolio_summary = build_executive_insights(moic_df, summary)
print("── Executive Insights ─────────────────────────────────────")
display(executive_insights)

print("\n── Portfolio Summary ───────────────────────────────────────")
display(portfolio_summary)

outputs = {
    "cleaned_installments.csv"   : cleaned,
    "excluded_rows.csv"          : excluded,
    "moic_table.csv"             : moic_df,
    "cohort_summary.csv"         : summary,
    "executive_insights.csv"     : executive_insights,
    "portfolio_summary.csv"      : portfolio_summary,
    "cumulative_moic_matrix.csv" : cum_matrix,
    "payment_moic_matrix.csv"    : pay_matrix,
}
for filename, df in outputs.items():
    write_csv(df, f"{out_path}/{filename}")
    print(f"  ✓  {filename:<38}  ({len(df):,} rows)")

── Cohort summary table ────────────────────────────────────


,Cohort,Asset_Count,Cohort_Total_Advance,Total_Collected,Max_Months_on_Book,Max_Cumulative_MOIC,Max_Net_MOIC,Avg_Payment_MOIC
0,2024-01,6,102945.0,119150.0,9,1.157414,0.157414,0.128602
1,2024-02,11,136095.0,181074.0,8,1.330497,0.330497,0.147833
2,2024-03,64,795665.0,1030222.0,12,1.294794,0.294794,0.099600
3,2024-04,32,368935.0,506270.0,11,1.372247,0.372247,0.114354
4,2024-05,16,224805.0,293500.0,9,1.305576,0.305576,0.130558
5,2024-06,32,447855.0,583578.0,21,1.303051,0.303051,0.059230
6,2024-07,20,286900.0,381188.0,9,1.328644,0.328644,0.132864
7,2024-08,28,325990.0,445617.0,16,1.366965,0.366965,0.080410
8,2024-09,37,408306.0,567657.0,17,1.390273,0.390273,0.077237
9,2024-10,37,461204.0,614090.0,15,1.331493,0.331493,0.083218



── Cumulative MOIC matrix (cohort × MOB) ───────────────────


,Cohort,MOB_0,MOB_1,MOB_2,MOB_3,MOB_4,MOB_5,MOB_6,MOB_7,MOB_8,...,MOB_12,MOB_13,MOB_14,MOB_15,MOB_16,MOB_17,MOB_18,MOB_19,MOB_20,MOB_21
0,2024-01,NaN,0.228258,0.462334,0.846811,0.970071,1.085113,1.133197,1.133197,1.133197,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-02,0.055256,0.284845,0.543363,0.749572,0.991690,1.201800,1.304537,1.304537,1.330497,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-03,0.097275,0.327693,0.556440,0.778335,0.977024,1.166996,1.210851,1.261189,1.276129,...,1.294794,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-04,0.100129,0.357055,0.575980,0.799959,1.029883,1.206936,1.276008,1.312413,1.328879,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-05,0.082649,0.317827,0.542741,0.741709,0.949774,1.194102,1.287605,1.296524,1.300305,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2024-06,0.135437,0.354228,0.589803,0.784051,0.972136,1.110911,1.201864,1.240196,1.250579,...,1.268174,1.270407,1.275989,1.277552,1.277552,1.282018,1.286483,1.292974,1.299032,1.303051
6,2024-07,0.140415,0.376978,0.607801,0.839826,1.031854,1.256239,1.269306,1.314793,1.314793,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2024-08,0.045379,0.288509,0.540458,0.760560,0.979183,1.196816,1.300798,1.317362,1.323406,...,1.328774,1.336289,1.354695,1.357763,1.366965,NaN,NaN,NaN,NaN,NaN
8,2024-09,0.078250,0.317265,0.544187,0.789508,0.988705,1.165454,1.297880,1.316248,1.339515,...,1.368231,1.378028,1.387824,1.387824,1.387824,1.390273,NaN,NaN,NaN,NaN
9,2024-10,0.088967,0.336602,0.598824,0.817630,0.990590,1.150996,1.222990,1.253296,1.255464,...,1.296715,1.307122,1.323774,1.331493,NaN,NaN,NaN,NaN,NaN,NaN



── Payment MOIC matrix (cohort × MOB) ─────────────────────


...


[3b] Building Executive Insights ...
── Executive Insights ─────────────────────────────────────


,Cohort,Recovery_Status,Seasoned_12M,Latest_MOB,Latest_Cumulative_MOIC,Months_to_Breakeven,Months_to_Target_1_3x,Collection_Share_Pct,Advance_Share_Pct,Max_Cumulative_MOIC
8,2024-09,Beyond target,True,17,1.390273,5,7,10.022193,9.083004,1.390273
3,2024-04,Beyond target,False,11,1.372247,4,7,8.938383,8.207173,1.372247
7,2024-08,Beyond target,True,16,1.366965,5,6,7.867532,7.251837,1.366965
9,2024-10,Beyond target,True,15,1.331493,5,13,10.841984,10.259751,1.331493
1,2024-02,Beyond target,False,8,1.330497,5,6,3.196928,3.027512,1.330497
6,2024-07,Beyond target,False,9,1.328644,4,7,6.730014,6.382257,1.328644
4,2024-05,Beyond target,False,9,1.305576,5,8,5.181850,5.000918,1.305576
5,2024-06,Beyond target,True,21,1.303051,5,21,10.303284,9.962794,1.303051
2,2024-03,Recovered,True,12,1.294794,5,<NA>,18.188948,17.700029,1.294794
12,2025-09,Recovered,True,12,1.249359,4,<NA>,0.215042,0.216872,1.249359



── Portfolio Summary ───────────────────────────────────────


,Metric,Value,Unit
0,Portfolio Recovery Rate,1.26,x
1,Recovered Cohort Share,95.28,% of collections
2,Top-3 Cohort Share,26.83,% of collections
3,Seasoned Cohorts (12M+),8.00,cohorts
4,Cohorts Above 1.0x,12.00,cohorts
5,Cohorts Above 1.3x,8.00,cohorts


  ✓  cleaned_installments.csv                (11,252 rows)
  ✓  excluded_rows.csv                       (4 rows)
  ✓  moic_table.csv                          (185 rows)
  ✓  cohort_summary.csv                      (15 rows)
  ✓  executive_insights.csv                  (15 rows)
  ✓  portfolio_summary.csv                   (6 rows)
  ✓  cumulative_moic_matrix.csv              (15 rows)
  ✓  payment_moic_matrix.csv                 (15 rows)


## Section 4 · MOIC Curve Chart

One vintage curve per origination cohort — X-axis: Months on Book, Y-axis: Cumulative MOIC. The figure is rendered inline below.

In [5]:
def plot_moic_curves(moic_df: pd.DataFrame, output_path: str) -> None:
    """
    Render one vintage MOIC curve per origination cohort.

    Chart spec
    ----------
    X-axis : Months on Book (integer)
    Y-axis : Cumulative MOIC (formatted as 0.00×)
    Lines  : one per origination cohort, colour-coded chronologically
    """
    cohorts = sorted(moic_df["Cohort"].unique())
    n       = len(cohorts)

    # Chronological colour gradient (oldest = deep blue, newest = bright yellow)
    PALETTE = plt.cm.turbo(np.linspace(0.10, 0.92, n))

    # ── Theme constants ───────────────────────────────────────────────────────
    BG       = "#0d1117"
    GRID_C   = "#21262d"
    LABEL_C  = "#e6edf3"
    TICK_C   = "#8b949e"
    ANNOT_C  = "#6e7681"

    fig, ax = plt.subplots(figsize=(16, 8))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)

    # ── Draw one curve per cohort ─────────────────────────────────────────────
    for i, cohort in enumerate(cohorts):
        sub = moic_df[moic_df["Cohort"] == cohort].sort_values("Months_on_Book")
        ax.plot(
            sub["Months_on_Book"],
            sub["Cumulative_MOIC"],
            marker="o",
            markersize=4.5,
            linewidth=2.0,
            color=PALETTE[i],
            label=cohort,
            alpha=0.92,
            zorder=3,
            markeredgewidth=0.6,
            markeredgecolor="#00000033",
        )

    # ── Reference lines ───────────────────────────────────────────────────────
    y_max = moic_df["Cumulative_MOIC"].max()
    for ref_y, ref_label in [
        (1.0, "Break-even  1.00×"),
        (1.3, "Target  1.30×"),
    ]:
        if ref_y <= y_max * 1.12:
            ax.axhline(
                ref_y, color="#cccccc", linestyle="--",
                linewidth=0.85, alpha=0.35, zorder=2,
            )
            ax.text(
                0.006, ref_y + 0.008,
                ref_label,
                transform=ax.get_yaxis_transform(),
                color=ANNOT_C, fontsize=8.5, va="bottom",
            )

    # ── Axis labels & title ───────────────────────────────────────────────────
    ax.set_xlabel("Months on Book (MOB)", color=LABEL_C, fontsize=12, labelpad=10)
    ax.set_ylabel("Cumulative MOIC", color=LABEL_C, fontsize=12, labelpad=10)
    ax.set_title(
        "SahajMobile — Vintage MOIC Curves by Origination Cohort",
        color="white", fontsize=14, fontweight="bold", pad=18,
    )

    # ── Tick formatting ───────────────────────────────────────────────────────
    ax.tick_params(colors=TICK_C, labelsize=9.5)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda y, _: f"{y:.2f}×")
    )
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.set_ylim(bottom=0)
    ax.set_xlim(left=0)

    # ── Grid & spines ─────────────────────────────────────────────────────────
    for spine in ax.spines.values():
        spine.set_edgecolor("#30363d")
    ax.grid(True, color=GRID_C, linestyle="--", linewidth=0.65, zorder=1)

    # ── Footer annotation ─────────────────────────────────────────────────────
    fig.text(
        0.5, 0.01,
        f"Only actual collections (Payment_Amount > 0) are included  |  "
        f"{n} origination cohorts  |  "
        f"Dashed lines: break-even (1.00×) and target (1.30×)",
        ha="center", color=ANNOT_C, fontsize=8,
    )

    # ── Legend (outside chart, right side) ───────────────────────────────────
    leg = ax.legend(
        title="Origination\nCohort",
        title_fontsize=9,
        fontsize=8.5,
        ncol=1,
        framealpha=0.25,
        edgecolor="#444c56",
        facecolor="#161b22",
        labelcolor="white",
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        borderaxespad=0,
        handlelength=1.8,
        handleheight=1.0,
    )
    leg.get_title().set_color(TICK_C)

    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.savefig(
        output_path, dpi=150, bbox_inches="tight",
        facecolor=fig.get_facecolor(),
    )
    plt.close(fig)
    print(f"  Saved → {output_path}")


def plot_net_moic_curves(moic_df: pd.DataFrame, output_path: str) -> None:
    """
    Render one Net MOIC curve per origination cohort (separate chart).

    Net MOIC = Cumulative MOIC − 1.00×  (returns above/below invested capital).

    Chart spec
    ----------
    X-axis : Months on Book (integer)
    Y-axis : Net MOIC (formatted as 0.00×)
    Lines  : one per origination cohort, colour-coded chronologically
    """
    cohorts = sorted(moic_df["Cohort"].unique())
    n       = len(cohorts)

    # Chronological colour gradient (oldest = deep blue, newest = bright yellow)
    PALETTE = plt.cm.turbo(np.linspace(0.10, 0.92, n))

    # ── Theme constants ───────────────────────────────────────────────────────
    BG       = "#0d1117"
    GRID_C   = "#21262d"
    LABEL_C  = "#e6edf3"
    TICK_C   = "#8b949e"
    ANNOT_C  = "#6e7681"

    fig, ax = plt.subplots(figsize=(16, 8))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)

    # ── Draw one curve per cohort ─────────────────────────────────────────────
    for i, cohort in enumerate(cohorts):
        sub = moic_df[moic_df["Cohort"] == cohort].sort_values("Months_on_Book")
        ax.plot(
            sub["Months_on_Book"],
            sub["Net_MOIC"],
            marker="o",
            markersize=4.5,
            linewidth=2.0,
            color=PALETTE[i],
            label=cohort,
            alpha=0.92,
            zorder=3,
            markeredgewidth=0.6,
            markeredgecolor="#00000033",
        )

    # ── Reference lines ───────────────────────────────────────────────────────
    y_max = moic_df["Net_MOIC"].max()
    y_min = moic_df["Net_MOIC"].min()
    for ref_y, ref_label in [
        (0.0, "Net break-even  0.00×"),
    ]:
        ax.axhline(
            ref_y, color="#cccccc", linestyle="--",
            linewidth=0.85, alpha=0.35, zorder=2,
        )
        ax.text(
            0.006, ref_y + 0.008,
            ref_label,
            transform=ax.get_yaxis_transform(),
            color=ANNOT_C, fontsize=8.5, va="bottom",
        )

    # ── Axis labels & title ───────────────────────────────────────────────────
    ax.set_xlabel("Months on Book (MOB)", color=LABEL_C, fontsize=12, labelpad=10)
    ax.set_ylabel("Net MOIC (Cumulative MOIC − 1.00×)", color=LABEL_C, fontsize=12, labelpad=10)
    ax.set_title(
        "SahajMobile — Net MOIC Curves by Origination Cohort",
        color="white", fontsize=14, fontweight="bold", pad=18,
    )

    # ── Tick formatting ───────────────────────────────────────────────────────
    ax.tick_params(colors=TICK_C, labelsize=9.5)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda y, _: f"{y:+.2f}×")
    )
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.set_ylim(bottom=min(0.0, y_min - 0.06), top=max(0.0, y_max) + 0.06)
    ax.set_xlim(left=0)

    # ── Grid & spines ─────────────────────────────────────────────────────────
    for spine in ax.spines.values():
        spine.set_edgecolor("#30363d")
    ax.grid(True, color=GRID_C, linestyle="--", linewidth=0.65, zorder=1)

    # ── Footer annotation ─────────────────────────────────────────────────────
    fig.text(
        0.5, 0.01,
        f"Only actual collections (Payment_Amount > 0) are included  |  "
        f"{n} origination cohorts  |  "
        f"Net MOIC = Cumulative MOIC − 1.00× (above 0.00× = profit above advance)",
        ha="center", color=ANNOT_C, fontsize=8,
    )

    # ── Legend (outside chart, right side) ───────────────────────────────────
    leg = ax.legend(
        title="Origination\nCohort",
        title_fontsize=9,
        fontsize=8.5,
        ncol=1,
        framealpha=0.25,
        edgecolor="#444c56",
        facecolor="#161b22",
        labelcolor="white",
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        borderaxespad=0,
        handlelength=1.8,
        handleheight=1.0,
    )
    leg.get_title().set_color(TICK_C)

    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.savefig(
        output_path, dpi=150, bbox_inches="tight",
        facecolor=fig.get_facecolor(),
    )
    plt.close(fig)
    print(f"  Saved → {output_path}")


def plot_executive_dashboard(
    moic_df: pd.DataFrame, cohort_summary: pd.DataFrame, output_path: str
) -> None:
    """
    Render an executive-style dashboard with curve, heatmap, ranking, and KPI panels.
    """
    cohorts = sorted(moic_df["Cohort"].unique())
    curve_matrix = (
        moic_df.pivot_table(
            index="Cohort",
            columns="Months_on_Book",
            values="Cumulative_MOIC",
            aggfunc="first",
        )
        .reindex(cohorts)
    )

    latest_rows = (
        moic_df.sort_values(["Cohort", "Months_on_Book"])
              .groupby("Cohort", as_index=False)
              .tail(1)
              [["Cohort", "Months_on_Book", "Cumulative_MOIC"]]
    )
    latest_rows = cohort_summary.merge(latest_rows, on="Cohort", how="left")
    latest_rows["Recovery_Spread"] = (
        latest_rows["Total_Collected"] - latest_rows["Cohort_Total_Advance"]
    )

    peak_row = cohort_summary.loc[cohort_summary["Max_Cumulative_MOIC"].idxmax()]
    recovery_rate = (
        cohort_summary["Total_Collected"].sum()
        / cohort_summary["Cohort_Total_Advance"].sum()
    )

    BG       = "#0b1020"
    PANEL_BG = "#111827"
    GRID_C   = "#293241"
    LABEL_C  = "#e5e7eb"
    TICK_C   = "#9ca3af"
    POS_C    = "#22c55e"
    NEG_C    = "#ef4444"

    fig = plt.figure(figsize=(21, 13), facecolor=BG)
    gs = fig.add_gridspec(
        2,
        2,
        width_ratios=(1.35, 1.0),
        height_ratios=(1.15, 0.85),
        wspace=0.16,
        hspace=0.20,
    )
    ax_curve = fig.add_subplot(gs[0, 0])
    ax_heat = fig.add_subplot(gs[0, 1])
    ax_rank = fig.add_subplot(gs[1, 0])
    ax_kpi = fig.add_subplot(gs[1, 1])

    for ax in (ax_curve, ax_heat, ax_rank, ax_kpi):
        ax.set_facecolor(PANEL_BG)
        for spine in ax.spines.values():
            spine.set_edgecolor("#334155")

    palette = plt.cm.turbo(np.linspace(0.10, 0.92, len(cohorts)))

    for i, cohort in enumerate(cohorts):
        cohort_slice = moic_df[moic_df["Cohort"] == cohort].sort_values("Months_on_Book")
        ax_curve.plot(
            cohort_slice["Months_on_Book"],
            cohort_slice["Cumulative_MOIC"],
            linewidth=2.2,
            marker="o",
            markersize=3.8,
            color=palette[i],
            alpha=0.94,
            label=cohort,
        )

    ax_curve.axhline(1.0, color="#94a3b8", linestyle="--", linewidth=1.0, alpha=0.6)
    ax_curve.axhline(1.3, color="#f59e0b", linestyle="--", linewidth=1.0, alpha=0.7)
    ax_curve.set_title(
        "Vintage Cumulative MOIC Curves",
        color=LABEL_C,
        fontsize=14,
        fontweight="bold",
        pad=12,
    )
    ax_curve.set_xlabel("Months on Book", color=LABEL_C, fontsize=11)
    ax_curve.set_ylabel("Cumulative MOIC", color=LABEL_C, fontsize=11)
    ax_curve.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:.2f}×"))
    ax_curve.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax_curve.tick_params(colors=TICK_C, labelsize=9)
    ax_curve.grid(True, color=GRID_C, linestyle="--", linewidth=0.6, alpha=0.8)
    ax_curve.legend(
        title="Cohort",
        fontsize=8,
        title_fontsize=9,
        framealpha=0.20,
        facecolor=PANEL_BG,
        edgecolor="#475569",
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
    )

    heat_data = np.ma.masked_invalid(curve_matrix.to_numpy(dtype=float))
    heat_map = ax_heat.imshow(
        heat_data,
        aspect="auto",
        cmap=plt.cm.magma,
        interpolation="nearest",
        vmin=0,
        vmax=np.nanmax(curve_matrix.to_numpy(dtype=float)),
    )
    ax_heat.set_title(
        "Cumulative MOIC Heatmap",
        color=LABEL_C,
        fontsize=14,
        fontweight="bold",
        pad=12,
    )
    ax_heat.set_xlabel("Months on Book", color=LABEL_C, fontsize=11)
    ax_heat.set_ylabel("Cohort", color=LABEL_C, fontsize=11)
    x_positions = np.arange(len(curve_matrix.columns))
    x_labels = [f"MOB {c}" for c in curve_matrix.columns]
    step = max(1, len(x_positions) // 10)
    ax_heat.set_xticks(x_positions[::step])
    ax_heat.set_xticklabels(x_labels[::step], rotation=45, ha="right", color=TICK_C, fontsize=8)
    ax_heat.set_yticks(np.arange(len(curve_matrix.index)))
    ax_heat.set_yticklabels(curve_matrix.index, color=TICK_C, fontsize=8)
    ax_heat.tick_params(length=0)
    cbar = fig.colorbar(heat_map, ax=ax_heat, fraction=0.046, pad=0.03)
    cbar.ax.tick_params(colors=TICK_C, labelsize=8)
    cbar.set_label("Cumulative MOIC", color=LABEL_C, fontsize=9)

    ranked = latest_rows.sort_values("Max_Cumulative_MOIC", ascending=True)
    bar_colors = [POS_C if value >= 1.0 else NEG_C for value in ranked["Max_Cumulative_MOIC"]]
    ax_rank.barh(ranked["Cohort"], ranked["Max_Cumulative_MOIC"], color=bar_colors, alpha=0.9)
    ax_rank.axvline(1.0, color="#cbd5e1", linestyle="--", linewidth=1.0, alpha=0.7)
    ax_rank.set_title(
        "Cohort Recovery Strength",
        color=LABEL_C,
        fontsize=14,
        fontweight="bold",
        pad=12,
    )
    ax_rank.set_xlabel("Maximum Cumulative MOIC", color=LABEL_C, fontsize=11)
    ax_rank.tick_params(colors=TICK_C, labelsize=9)
    ax_rank.grid(True, axis="x", color=GRID_C, linestyle="--", linewidth=0.6, alpha=0.8)
    for idx, value in enumerate(ranked["Max_Cumulative_MOIC"]):
        ax_rank.text(value + 0.015, idx, f"{value:.2f}×", va="center", color=LABEL_C, fontsize=8)

    ax_kpi.axis("off")
    ax_kpi.text(
        0.00,
        0.98,
        "Executive Cohort Snapshot",
        transform=ax_kpi.transAxes,
        fontsize=15,
        fontweight="bold",
        color=LABEL_C,
        va="top",
    )
    kpi_box = dict(boxstyle="round,pad=0.55", facecolor="#0f172a", edgecolor="#334155", alpha=0.98)
    ax_kpi.text(0.02, 0.82, f"Cohorts analyzed\n{len(cohorts)}", transform=ax_kpi.transAxes, color=LABEL_C, fontsize=12, bbox=kpi_box)
    ax_kpi.text(0.36, 0.82, f"Recovery rate\n{recovery_rate:.2f}×", transform=ax_kpi.transAxes, color=LABEL_C, fontsize=12, bbox=kpi_box)
    ax_kpi.text(0.70, 0.82, f"Best cohort\n{peak_row['Cohort']} ({peak_row['Max_Cumulative_MOIC']:.2f}×)", transform=ax_kpi.transAxes, color=LABEL_C, fontsize=12, bbox=kpi_box)
    top_collected_cohort = latest_rows.sort_values('Total_Collected', ascending=False).iloc[0]['Cohort']
    ax_kpi.text(0.02, 0.48, f"Top collected cohort\n{top_collected_cohort}", transform=ax_kpi.transAxes, color=LABEL_C, fontsize=12, bbox=kpi_box)
    ax_kpi.text(0.36, 0.48, f"Average max MOIC\n{cohort_summary['Max_Cumulative_MOIC'].mean():.2f}×", transform=ax_kpi.transAxes, color=LABEL_C, fontsize=12, bbox=kpi_box)
    ax_kpi.text(0.70, 0.48, f"Peak month on book\n{int(cohort_summary['Max_Months_on_Book'].max())}", transform=ax_kpi.transAxes, color=LABEL_C, fontsize=12, bbox=kpi_box)
    ax_kpi.text(
        0.02,
        0.15,
        "The dashboard combines vintage curves, cohort density, and ranking signals to support an executive readout.",
        transform=ax_kpi.transAxes,
        color=TICK_C,
        fontsize=10,
        wrap=True,
    )

    fig.suptitle(
        "SahajMobile Cohort MOIC Executive Dashboard",
        color="white",
        fontsize=18,
        fontweight="bold",
        y=0.98,
    )
    fig.text(
        0.5,
        0.01,
        "Curves show cumulative collection build-up; heatmap shows cohort aging; ranking and KPIs support rapid portfolio review.",
        ha="center",
        color=TICK_C,
        fontsize=9,
    )

    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    plt.savefig(
        output_path,
        dpi=160,
        bbox_inches="tight",
        facecolor=fig.get_facecolor(),
    )
    plt.close(fig)
    print(f"  Saved → {output_path}")


plot_moic_curves(moic_df, f"{out_path}/moic_curve.png")
plot_net_moic_curves(moic_df, f"{out_path}/net_moic_curve.png")
plot_executive_dashboard(moic_df, summary, f"{out_path}/moic_dashboard.png")

  Saved → /Users/dipro/Task Sahaj Mobile/outputs/moic_curve.png


  Saved → /Users/dipro/Task Sahaj Mobile/outputs/net_moic_curve.png


  Saved → /Users/dipro/Task Sahaj Mobile/outputs/moic_dashboard.png
